### **Fase 0**: Import Generico

In [ ]:
import os
import sys
import importlib
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
else:
    print("Drive già montato, salto il mount.")

sys.path.append('/content/drive/MyDrive/progetto_immunogold/scripts')
importlib.invalidate_caches()

import common
importlib.reload(common)
from common import *

print(f"Device attivo: {device}")
print(paths)
log('Notebook avviato correttamente')

Drive già montato, salto il mount.
Device attivo: cpu
{'raw_images': '/content/drive/MyDrive/progetto_immunogold/raw_images', 'annotations': '/content/drive/MyDrive/progetto_immunogold/annotations', 'patches_images': '/content/drive/MyDrive/progetto_immunogold/patches/images', 'patches_coordinates': '/content/drive/MyDrive/progetto_immunogold/patches/coordinates', 'patches_density': '/content/drive/MyDrive/progetto_immunogold/patches/density_maps', 'checkpoints': '/content/drive/MyDrive/progetto_immunogold/checkpoints', 'logs': '/content/drive/MyDrive/progetto_immunogold/logs'}
[2026-09-17 09:44] Notebook avviato correttamente


- Controllo collisioni tra nominativi

In [ ]:
import pandas as pd

path_progetto = '/content/drive/MyDrive/progetto_immunogold'
calib = pd.read_csv(f'{path_progetto}/annotations/calibrazione.csv')

collisioni = calib.groupby("filename")["esperimento"].nunique()
collisioni = collisioni[collisioni > 1]

print(f"Filename duplicati tra esperimenti diversi: {len(collisioni)}")
print(collisioni)

Filename duplicati tra esperimenti diversi: 0
Series([], Name: esperimento, dtype: int64)


### **Fase 1**: Import librerie (gaussian_filter - serve a creare la campana gaussiana attorno a sigma)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.ndimage import gaussian_filter

from common import paths, log, device

### **Fase 2**: Calcolo sigma per classe di calibrazione e salvataggio dei valori nella cartella *annotations*

In [ ]:
def calcola_sigma_per_classe(path_calibrazione, path_statistiche_raggio_per_classe,
                              col_classe="classe_calibrazione",
                              col_nm_px="nm_per_pixel",
                              col_raggio="raggio_medio_nm",
                              col_cv="cv_percento"):

    calib = pd.read_csv(path_calibrazione)
    raggio = pd.read_csv(path_statistiche_raggio_per_classe)

    n_valori_distinti = calib.groupby(col_classe)[col_nm_px].nunique()
    incoerenti = n_valori_distinti[n_valori_distinti > 1]
    if not incoerenti.empty:
        log(f"ATTENZIONE - classi con più di un valore di nm_per_pixel al loro interno: {list(incoerenti.index)}")

    agg = calib.groupby(col_classe)[col_nm_px].first().reset_index()

    df = agg.merge(raggio, on=col_classe, how="inner")
    df["sigma_px"] = df[col_raggio] / df[col_nm_px]

    cols = [col_classe, col_nm_px, col_raggio, "sigma_px"]
    if col_cv in df.columns:
        cols.insert(-1, col_cv)

    out = df[cols].sort_values(col_classe).reset_index(drop=True)

    # Segnala classi con CV% del raggio elevato: sigma medio per classe
    # è un'approssimazione più grossolana in questi casi
    if col_cv in out.columns:
        alta_variabilita = out[out[col_cv] > 30]  # soglia indicativa, da tarare
        if not alta_variabilita.empty:
            log(f"ATTENZIONE - classi con CV% del raggio elevato (verifica visiva consigliata): {list(alta_variabilita[col_classe])}")

    return out

sigma_df = calcola_sigma_per_classe(
    os.path.join(paths['annotations'], 'calibrazione.csv'),
    os.path.join(paths['annotations'], 'statistiche_raggio_per_classe.csv'),
)

display(sigma_df)

sigma_df.to_csv(os.path.join(paths['annotations'], 'sigma_per_classe.csv'), index=False)

[2026-09-17 09:45] ATTENZIONE - classi con CV% del raggio elevato (verifica visiva consigliata): ['classe_7']


,classe_calibrazione,nm_per_pixel,raggio_medio_nm,cv_percento,sigma_px
0,classe_1,0.450,4.96,29.8,11.022222
1,classe_10,5.630,28.24,28.7,5.015986
2,classe_2,0.560,4.91,20.7,8.767857
3,classe_3,0.670,6.03,29.5,9.000000
4,classe_4,0.845,6.17,29.7,7.301775
5,classe_5,1.125,9.23,28.5,8.204444
6,classe_6,1.405,9.06,12.7,6.448399
7,classe_7,1.680,10.35,32.6,6.160714
8,classe_8,2.110,8.55,19.4,4.052133
9,classe_9,2.820,9.40,24.1,3.333333


### **Fase 3**: Generazione density map

In [ ]:
def genera_density_map(patch_shape, punti_xy, sigma_px, mode="constant"):
    """
    patch_shape : (H, W) dimensioni della patch
    punti_xy    : lista/array di (x, y) coordinate LOCALI dei puntini nella patch
                  (già ricalcolate rispetto all'offset del ritaglio, Fase 3)
    sigma_px    : sigma della gaussiana in pixel, per la classe di calibrazione
                  a cui appartiene l'immagine sorgente della patch
    mode        : 'constant' (default) tratta i bordi patch come vuoti;
                  ragionevole dato l'overlap di 32px usato in Fase 3,
                  che riduce il rischio di perdita di massa vicino ai bordi
                  per punti realmente centrali nella patch
    """
    density = np.zeros(patch_shape, dtype=np.float32)

    for x, y in punti_xy:
        xi, yi = int(round(x)), int(round(y))
        if 0 <= yi < patch_shape[0] and 0 <= xi < patch_shape[1]:
            density[yi, xi] += 1.0

    if len(punti_xy) > 0:
        density = gaussian_filter(density, sigma=sigma_px, mode=mode)

    return density

### **Fase 4**: Verifica che la somma della density map corrisponda al numero di puntini annotati nella patch (entro tolleranza tol, relativa).

In [ ]:
def verifica_sanita(density_map, n_punti_attesi, tol=0.05):

    somma = float(density_map.sum())
    n_punti_attesi = max(n_punti_attesi, 0)
    errore_relativo = abs(somma - n_punti_attesi) / n_punti_attesi if n_punti_attesi > 0 else 0.0
    ok = errore_relativo <= tol

    return {
        "somma_mappa": somma,
        "n_punti_attesi": n_punti_attesi,
        "errore_relativo": errore_relativo,
        "ok": ok,
    }

### **Fase 5**: Mappatura patch --> associazione del sigma corretto ad ogni patch

In [ ]:
def estrai_nome_immagine_da_patch(nome_file_patch):
    """
    Naming reale confermato sui dati: {nome_immagine_originale}_x{X}_y{Y}.png
    (coordinate dell'offset di ritaglio della patch, non un letterale '_patch').
    """
    match = re.match(r"(.+)_x\d+_y\d+$", Path(nome_file_patch).stem)
    if match is None:
        raise ValueError(f"Impossibile estrarre nome immagine da: {nome_file_patch}")
    return match.group(1)


def costruisci_mappa_patch_sigma(cartella_patches, path_calibrazione, sigma_per_classe_df,
                                  estensione="*.png",
                                  col_chiave_calibrazione="filename",
                                  col_classe="classe_calibrazione"):
    """
    cartella_patches        : path a patches/images/
    path_calibrazione       : annotations/calibrazione.csv — contiene già,
                               per ogni immagine originale, filename/filepath
                               e classe_calibrazione: nessun file aggiuntivo
                               da creare.
    sigma_per_classe_df     : output di calcola_sigma_per_classe()
    col_chiave_calibrazione : colonna di calibrazione.csv su cui fare match
                               con l'id immagine estratto dal nome patch —
                               'filename' o 'filepath'. ATTENZIONE: se i
                               filename possono ripetersi tra esperimenti
                               diversi, usa 'filepath' (o assicurati che il
                               nome della patch porti con sé l'informazione
                               di esperimento/data) per evitare match errati.

    Ritorna un DataFrame: patch_path, id_immagine, classe_calibrazione, sigma_px
    Nessuna riorganizzazione fisica delle patch richiesta.
    """
    calib = pd.read_csv(path_calibrazione)

    # calibrazione.csv ha filename CON estensione (es. '...22440.jpg'),
    # ma le patch sono .png e id_immagine è estratto senza estensione:
    # si confrontano entrambi i lati sullo "stem" per evitare mismatch.
    calib["_chiave_match"] = calib[col_chiave_calibrazione].apply(lambda s: Path(s).stem)

    patch_paths = sorted(Path(cartella_patches).glob(estensione))
    righe = [
        {"patch_path": str(p), "id_immagine": estrai_nome_immagine_da_patch(p.name)}
        for p in patch_paths
    ]
    df_patch = pd.DataFrame(righe)

    df_patch = df_patch.merge(
        calib[["_chiave_match", col_classe]],
        left_on="id_immagine", right_on="_chiave_match", how="left"
    ).drop(columns="_chiave_match")

    mancanti = df_patch[df_patch[col_classe].isna()]
    if not mancanti.empty:
        log(f"ATTENZIONE - {len(mancanti)} patch senza classe di calibrazione associata (controlla naming/match)")

    df_patch = df_patch.merge(
        sigma_per_classe_df[[col_classe, "sigma_px"]], on=col_classe, how="left"
    )

    return df_patch

### **Fase 6**: Controlli

In [ ]:
mappa_patch = costruisci_mappa_patch_sigma(
    paths['patches_images'],
    os.path.join(paths['annotations'], 'calibrazione.csv'),
    sigma_df,  # quello già calcolato nella cella Fase 2, non serve rifarlo
    col_chiave_calibrazione="filename",
)
mappa_patch.head()

,patch_path,id_immagine,classe_calibrazione,sigma_px
0,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X10k_JEM-1400 Flash_22440,classe_7,6.160714
1,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X10k_JEM-1400 Flash_22440,classe_7,6.160714
2,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X10k_JEM-1400 Flash_22440,classe_7,6.160714
3,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X10k_JEM-1400 Flash_22440,classe_7,6.160714
4,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X10k_JEM-1400 Flash_22440,classe_7,6.160714


In [ ]:
# 1. Nessuna patch è rimasta senza classe o senza sigma
print("Patch totali:", len(mappa_patch))
print("Patch senza classe:", mappa_patch['classe_calibrazione'].isna().sum())
print("Patch senza sigma:", mappa_patch['sigma_px'].isna().sum())

# 2. Il numero di righe coincide con i file fisici su disco
n_file_reali = len(list(Path(paths['patches_images']).glob('*.png')))
print("File fisici in patches/images/:", n_file_reali)
assert len(mappa_patch) == n_file_reali, "Mismatch tra righe mappa_patch e file su disco"

# 3. Solo le 10 classi attese, nessun typo o valore anomalo
print(sorted(mappa_patch['classe_calibrazione'].dropna().unique()))

# 4. Nessuna patch duplicata nel risultato
assert mappa_patch['patch_path'].is_unique, "patch_path duplicati trovati"

# 5. Distribuzione patch per classe e per immagine sorgente
display(mappa_patch['classe_calibrazione'].value_counts())
display(mappa_patch['id_immagine'].value_counts())

Patch totali: 1342
Patch senza classe: 0
Patch senza sigma: 0
File fisici in patches/images/: 1342
['classe_1', 'classe_10', 'classe_2', 'classe_3', 'classe_4', 'classe_5', 'classe_6', 'classe_7', 'classe_8', 'classe_9']


,count
classe_calibrazione,
classe_5,234
classe_7,162
classe_6,162
classe_3,162
classe_2,162
classe_9,153
classe_1,81
classe_10,81
classe_8,81


,count
id_immagine,
SA-MAG_X10k_JEM-1400 Flash_22440,81
SA-MAG_X10k_JEM-1400 Flash_22480,81
SA-MAG_X12k_JEM-1400 Flash_22429,81
SA-MAG_X12k_JEM-1400 Flash_22484,81
SA-MAG_X15k_JEM-1400 Flash_22478,81
SA-MAG_X25k_JEM-1400 Flash_22433,81
SA-MAG_X15k_JEM-1400 Flash_22598,81
SA-MAG_X3000_JEM-1400 Flash_22441,81
SA-MAG_X25k_JEM-1400 Flash_22438,81


In [ ]:
cartella_punti = paths['patches_coordinates']

def path_coordinate_per_patch(patch_path):
    nome_patch = Path(patch_path).stem
    return os.path.join(cartella_punti, f"{nome_patch}_points.csv")

mappa_patch['ha_coordinate'] = mappa_patch['patch_path'].apply(
    lambda p: os.path.exists(path_coordinate_per_patch(p))
)

print("Patch con CSV coordinate disponibile:", mappa_patch['ha_coordinate'].sum())
print("Patch ancora senza annotazione:", (~mappa_patch['ha_coordinate']).sum())

Patch con CSV coordinate disponibile: 1342
Patch ancora senza annotazione: 0


### **Fase 7**: Generazione e salvataggio delle density map

In [ ]:
from PIL import Image

def path_coordinate_per_patch(patch_path):
    nome_patch = Path(patch_path).stem
    return os.path.join(paths['patches_coordinates'], f"{nome_patch}_points.csv")

def path_output_density_map(patch_path):
    nome_patch = Path(patch_path).stem
    return os.path.join(paths['patches_density'], f"{nome_patch}.npy")

os.makedirs(paths['patches_density'], exist_ok=True)

risultati_sanita = []
patch_saltate = 0
patch_gia_presenti = 0
patch_nuove_generate = 0
immagini_processate = set()
immagini_saltate = set()

for _, riga in mappa_patch.iterrows():
    patch_path = riga['patch_path']
    sigma_px = riga['sigma_px']
    id_immagine = riga['id_immagine']

    path_output = path_output_density_map(patch_path)
    if os.path.exists(path_output):
        patch_gia_presenti += 1
        immagini_processate.add(id_immagine)
        continue  # density map già generata in una run precedente, non rifare il lavoro

    path_coord = path_coordinate_per_patch(patch_path)
    if not os.path.exists(path_coord):
        patch_saltate += 1
        immagini_saltate.add(id_immagine)
        continue  # patch non ancora annotata, si salta per ora

    with Image.open(patch_path) as img:
        patch_shape = np.array(img).shape[:2]

    coord_df = pd.read_csv(path_coord)
    punti_xy = list(zip(coord_df['x_local'], coord_df['y_local']))

    density_map = genera_density_map(patch_shape, punti_xy, sigma_px)

    esito = verifica_sanita(density_map, n_punti_attesi=len(punti_xy))
    esito['patch_path'] = patch_path
    risultati_sanita.append(esito)

    np.save(path_output, density_map)
    patch_nuove_generate += 1
    immagini_processate.add(id_immagine)

immagini_completamente_saltate = immagini_saltate - immagini_processate

print(f"Patch già presenti (non rigenerate): {patch_gia_presenti}")
print(f"Patch nuove generate in questa run: {patch_nuove_generate}")
print(f"Patch totali con density map disponibile: {patch_gia_presenti + patch_nuove_generate} / {len(mappa_patch)}")
print(f"Patch saltate (nessuna annotazione trovata): {patch_saltate}")
print(f"Immagini con almeno una density map disponibile: {len(immagini_processate)}")
print(f"Immagini completamente non annotate: {len(immagini_completamente_saltate)}")

log(f"Fase 7 completata: {patch_nuove_generate} nuove, {patch_gia_presenti} già presenti, "
    f"{patch_saltate} saltate, {len(immagini_processate)} immagini coperte, "
    f"{len(immagini_completamente_saltate)} immagini ancora non annotate")

sanita_df = pd.DataFrame(risultati_sanita)
if not sanita_df.empty:
    display(sanita_df.describe())
else:
    print("Nessuna nuova density map generata in questa run: nessuna statistica di sanità da mostrare (i risultati della run precedente restano validi e non vengono ricalcolati).")

Patch già presenti (non rigenerate): 1342
Patch nuove generate in questa run: 0
Patch totali con density map disponibile: 1342 / 1342
Patch saltate (nessuna annotazione trovata): 0
Immagini con almeno una density map disponibile: 17
Immagini completamente non annotate: 0
[2026-09-17 09:45] Fase 7 completata: 0 nuove, 1342 già presenti, 0 saltate, 17 immagini coperte, 0 immagini ancora non annotate
Nessuna nuova density map generata in questa run: nessuna statistica di sanità da mostrare (i risultati della run precedente restano validi e non vengono ricalcolati).


In [ ]:
path_sanita_storico = os.path.join(paths['annotations'], 'sanita_storico.csv')

if not os.path.exists(path_sanita_storico):
    print("Nessuno storico esistente: ricostruisco dalle .npy già salvate su disco.")
    risultati_ricostruiti = []
    for _, riga in mappa_patch.iterrows():
        patch_path = riga['patch_path']
        id_immagine = riga['id_immagine']
        path_output = path_output_density_map(patch_path)
        path_coord = path_coordinate_per_patch(patch_path)
        if not os.path.exists(path_output) or not os.path.exists(path_coord):
            continue
        density_map = np.load(path_output)
        coord_df = pd.read_csv(path_coord)
        esito = verifica_sanita(density_map, n_punti_attesi=len(coord_df))
        esito['patch_path'] = patch_path
        esito['id_immagine'] = id_immagine
        esito['data_generazione'] = 'ricostruito_da_npy'
        risultati_ricostruiti.append(esito)
    sanita_storico_df = pd.DataFrame(risultati_ricostruiti)
    sanita_storico_df.to_csv(path_sanita_storico, index=False)
    print(f"Storico ricostruito: {len(sanita_storico_df)} righe.")
else:
    sanita_storico_df = pd.read_csv(path_sanita_storico)
    if risultati_sanita:  # eventuali nuove righe generate nella Fase 7 di questa run
        run_df = pd.DataFrame(risultati_sanita)
        sanita_storico_df = pd.concat([sanita_storico_df, run_df], ignore_index=True)
        sanita_storico_df = sanita_storico_df.drop_duplicates(subset='patch_path', keep='last')
        sanita_storico_df.to_csv(path_sanita_storico, index=False)
        print(f"Storico aggiornato con {len(run_df)} nuove righe. Totale: {len(sanita_storico_df)}.")
    else:
        print("Nessuna patch nuova in questa run: storico invariato.")

sanita_df = sanita_storico_df
print(f"\nStorico completo ({len(sanita_df)} patch):")
display(sanita_df.describe())

Nessuno storico esistente: ricostruisco dalle .npy già salvate su disco.
Storico ricostruito: 1342 righe.

Storico completo (1342 patch):


,somma_mappa,n_punti_attesi,errore_relativo
count,1342.000000,1342.000000,1342.000000
mean,5.691953,5.904620,0.033456
std,8.092829,8.298845,0.072396
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,2.772566,3.000000,0.002180
75%,7.740880,8.000000,0.041152
max,66.100929,67.000000,1.000000


### **Controlli**
1 - False: errore di tolleranza > 5 %

2 - Mostra le patches relative alle density maps con errore relativo più elevato

In [ ]:
print(sanita_df['ok'].value_counts())

ok
True     1056
False     286
Name: count, dtype: int64


In [ ]:
falliti = sanita_df[~sanita_df['ok']].sort_values('errore_relativo', ascending=False)
display(falliti.head(10))

,somma_mappa,n_punti_attesi,errore_relativo,ok,patch_path,id_immagine,data_generazione
943,0.000000,1,1.000000,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X30k_JEM-1400 Flash_22426,ricostruito_da_npy
1251,0.559844,2,0.720078,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X6000_JEM-1400 Flash_22467,ricostruito_da_npy
337,0.572581,2,0.713709,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X15k_JEM-1400 Flash_22427,ricostruito_da_npy
477,0.464115,1,0.535885,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X15k_JEM-1400 Flash_22598,ricostruito_da_npy
443,0.991324,2,0.504338,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X15k_JEM-1400 Flash_22478,ricostruito_da_npy
1011,1.000000,2,0.500000,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X30k_JEM-1400 Flash_22562,ricostruito_da_npy
408,1.541980,3,0.486007,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X15k_JEM-1400 Flash_22478,ricostruito_da_npy
159,0.532379,1,0.467621,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X10k_JEM-1400 Flash_22480,ricostruito_da_npy
949,0.554029,1,0.445971,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X30k_JEM-1400 Flash_22562,ricostruito_da_npy
1242,0.559844,1,0.440156,False,/content/drive/MyDrive/progetto_immunogold/pat...,SA-MAG_X6000_JEM-1400 Flash_22467,ricostruito_da_npy


Salvataggio *mappa_patch*



In [ ]:
path_mappa_patch = os.path.join(paths['annotations'], 'mappa_patch.csv')
mappa_patch.to_csv(path_mappa_patch, index=False)
log(f"mappa_patch salvata in {path_mappa_patch}")

[2026-09-17 10:07] mappa_patch salvata in /content/drive/MyDrive/progetto_immunogold/annotations/mappa_patch.csv
